# Custodian Guardian Layer — structure-preserving de-identification demo

*Companion to* **"Surrogate Substitution Preserves PHI Detectability: A Multi-Detector Equivalence Study."**

This notebook runs the paper's **case-study method end-to-end on a small sample**, so anyone can reproduce the core idea in a few minutes:

1. Take clinical / PII text with known PHI.
2. Apply the **Custodian Guardian Layer `transform`** (our method): replace each PHI value with a realistic *same-type surrogate*, leaving everything else byte-identical.
3. Check that a downstream PHI detector (Microsoft Presidio, CPU, free) **still finds the surrogate** — i.e. the substitution is *structure-preserving*.
4. Compute masked-span recall retention and run the **TOST equivalence** test (the statistic the paper leads with).

**What you need:** a Custodian API key for step 2 (get one at the Custodian docs). Steps 3–4 run with no key.

Full 11-detector × 7-benchmark results, data subsets, and scoring code: **https://custodianai.pages.dev/code**

## 1. Install dependencies

In [ ]:
%pip install -q custodian-labs presidio-analyzer presidio-anonymizer faker numpy scipy
!python -m spacy download en_core_web_lg -q
print('deps installed — restart runtime if spaCy was just installed, then continue.')

## 2. Your Custodian API key
Entered with `getpass`, so it is never printed or saved in the notebook.

In [ ]:
import os, getpass
os.environ['CUSTODIAN_SDK_API_KEY'] = getpass.getpass('Custodian API key: ')
from custodian_labs import GuardianLayer
guardian = GuardianLayer()
print('Guardian Layer client ready.')

## 3. Sample data (synthetic — no real patient data)
Each item lists the PHI values by substring; we locate character offsets programmatically so there are no hand-typed indices.

In [ ]:
SAMPLES = [
    {'text': 'What is the latest treatment protocol for a 34-year-old female diagnosed with MS like Anna S., previously treated at Methodist Hospital on April 12, 2023?',
     'phi': [('Anna S.', 'NAME'), ('Methodist Hospital', 'LOCATION'), ('April 12, 2023', 'DATE')]},
    {'text': 'Rec mgmt of 70yo M w/ CHF, seen by Dr. John L. at Mt. Sinai on Feb 21, 2023; looking for alt tx options.',
     'phi': [('John L.', 'NAME'), ('Mt. Sinai', 'LOCATION'), ('Feb 21, 2023', 'DATE')]},
    {'text': 'Follow-up for Maria Gomez, MRN 4471982, discharged from Cedars-Sinai on 03/14/2022; contact maria.g@example.com.',
     'phi': [('Maria Gomez', 'NAME'), ('4471982', 'ID'), ('Cedars-Sinai', 'LOCATION'), ('03/14/2022', 'DATE'), ('maria.g@example.com', 'EMAIL')]},
    {'text': 'Patient Robert Chen, DOB 08/09/1961, referred by Dr. Patel at Massachusetts General for a cardiac workup.',
     'phi': [('Robert Chen', 'NAME'), ('08/09/1961', 'DATE'), ('Patel', 'NAME'), ('Massachusetts General', 'LOCATION')]},
]

def spans(item):
    out = []
    for sub, lab in item['phi']:
        i = item['text'].find(sub)
        if i >= 0:
            out.append({'start': i, 'end': i + len(sub), 'label': lab, 'text': sub})
    return out

print(f'{len(SAMPLES)} sample documents, {sum(len(s["phi"]) for s in SAMPLES)} PHI spans.')

## 4. Apply our method: the `transform` (real value → same-type surrogate)

In [ ]:
def transform(text):
    r = guardian.deidentify_text_outputs(text, masking_type='transform', pii_entities=['ALL'])
    outs = r.outputs or []
    return outs[0].text if outs else text

for s in SAMPLES:
    s['xfrm'] = transform(s['text'])
    print('ORIG :', s['text'])
    print('XFRM :', s['xfrm'])
    print('-' * 100)

## 5. Does the surrogate stay detectable? (Microsoft Presidio, CPU)
We score PHI detection on the original and on the transformed text, and report how many PHI values Presidio still recovers. If detection holds, the substitution is *structure-preserving*.

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

provider = NlpEngineProvider(nlp_configuration={'nlp_engine_name': 'spacy',
    'models': [{'lang_code': 'en', 'model_name': 'en_core_web_lg'}]})
analyzer = AnalyzerEngine(nlp_engine=provider.create_engine(), supported_languages=['en'])

def detect(text):
    return [(r.start, r.end) for r in analyzer.analyze(text=text, language='en')]

def overlap(a, b):
    return not (a[1] <= b[0] or a[0] >= b[1])

orig_found = orig_total = 0
for s in SAMPLES:
    preds = detect(s['text'])
    for g in spans(s):
        orig_total += 1
        if any(overlap((g['start'], g['end']), p) for p in preds):
            orig_found += 1
print(f'Presidio recall on ORIGINAL PHI: {orig_found}/{orig_total} = {100*orig_found/orig_total:.0f}%')
print('\nPHI entities Presidio finds, original vs transformed (document level):')
for s in SAMPLES:
    print(f"  orig={len(detect(s['text'])):2d}   transformed={len(detect(s['xfrm'])):2d}   |  {s['text'][:55]}...")
print('\nSimilar counts => the surrogate carries the same detectable structure as the original.')

## 6. The statistic the paper leads with: TOST equivalence
Null-hypothesis tests are uninformative at large N (any tiny difference becomes "significant"). TOST instead asks whether the effect is provably *inside* a margin Δ. Below is the exact function from the paper's `analyze_equivalence.py`; the full-scale result over 57,112 masked spans is recall 76.1%→74.9%, TOST Δ=2pt **p≈3×10⁻⁹** (equivalent).

In [ ]:
import math

def tost_paired(b, c, n, delta):
    """TOST for a paired difference of proportions d=(b-c)/n at margin `delta`.
    b = found-originally-but-not-after, c = found-after-but-not-originally.
    Returns (difference_pct, p_value, equivalent?)."""
    d = (b - c) / n
    var = ((b + c) - (b - c) ** 2 / n) / (n ** 2)
    se = math.sqrt(var) if var > 0 else 1e-12
    p1 = 0.5 * math.erfc((d + delta) / se / math.sqrt(2))   # H0: mu <= -delta
    p2 = 0.5 * math.erfc(-(d - delta) / se / math.sqrt(2))  # H0: mu >= +delta
    p = max(p1, p2)
    return d * 100, p, p < 0.05

# Full-scale pooled counts from the paper (see custodianai.pages.dev/code):
b, c, n = 3300, 2612, 57112       # lost, gained, masked spans
diff, p, equiv = tost_paired(b, c, n, delta=0.02)
print(f'recall change (orig - transformed): {diff:+.2f} pts')
print(f'TOST at margin +-2 pts:  p = {p:.1e}  ->  {"EQUIVALENT" if equiv else "not equivalent"}')

## 7. Where to go next
- **Full evaluation** (11 detectors × 7 benchmarks × 7 languages), data subsets, and all scoring scripts: **https://custodianai.pages.dev/code**
- **Paper (PDF):** https://custodianai.pages.dev/paper.pdf
- **Interactive dashboard:** https://custodianai.pages.dev

The residual ~1.2-pt gap in the full study traces to surrogate-generation quality (truncation, salience loss, `x`-masking), not to detectors getting worse at PHI — a fixable, transform-side property.